In [14]:
import numpy as np
import pandas as pd
import keras
from keras import ops
from keras import layers
import keras_hub
from sklearn.model_selection import train_test_split

In [ ]:
def load_data_and_encode(dataset_path, nuc_to_idx, max_rows):
    """
    Read the Ribonanza CSV file, keep the sequences measured with both
    experiments (DMS_MaP and 2A3_MaP) and encode each nucleotide sequence
    as a list of integer indices.

    Parameters
    ----------
    dataset_path : str
        Path to the CSV file (train_data_QUICK_START.csv). Each sequence
        appears on two rows, one per experiment_type.
    nuc_to_idx : dict
        Mapping from nucleotide character (A/C/G/U) to integer index.
    max_rows : int or None
        Number of sequences to keep (useful to test quickly).
        None keeps all the sequences.

    Returns
    -------
    df_dms : pd.DataFrame
        Rows of the DMS_MaP experiment, sorted by sequence_id.
    df_2a3 : pd.DataFrame
        Rows of the 2A3_MaP experiment, sorted by sequence_id.
        Row i of df_dms and row i of df_2a3 are the same sequence.
    sequences : list of str
        Nucleotide sequences as strings.
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    """
    sequence_dataset = pd.read_csv(dataset_path)

    # We split the two experiments in two tables
    df_dms = sequence_dataset[sequence_dataset["experiment_type"] == "DMS_MaP"]
    df_2a3 = sequence_dataset[sequence_dataset["experiment_type"] == "2A3_MaP"]

    # We keep only the sequences which have the two measures
    ids_dms = set(df_dms["sequence_id"])
    ids_2a3 = set(df_2a3["sequence_id"])
    common_ids = sorted(ids_dms & ids_2a3)

    if max_rows is not None:
        common_ids = common_ids[:max_rows]

    # We sort by sequence_id so that the two tables are in the same order
    df_dms = df_dms[df_dms["sequence_id"].isin(common_ids)]
    df_dms = df_dms.drop_duplicates(subset="sequence_id")
    df_dms = df_dms.sort_values("sequence_id")

    df_2a3 = df_2a3[df_2a3["sequence_id"].isin(common_ids)]
    df_2a3 = df_2a3.drop_duplicates(subset="sequence_id")
    df_2a3 = df_2a3.sort_values("sequence_id")

    # The sequences are the same in the two tables
    sequences = df_dms["sequence"].astype(str).tolist()

    # We want to turn letters into numbers before putting them into a vector
    encoded_seqs = []

    # We map each letter to its number
    for seq in sequences:
        encoded_seq = []

        for c in seq:
            encoded_seq.append(nuc_to_idx.get(c, 0))

        encoded_seqs.append(encoded_seq)

    return df_dms, df_2a3, sequences, encoded_seqs

In [ ]:
def build_targets(df_dms, df_2a3, maxlen):
    """
    Build the target array with the two reactivities of each position.

    Parameters
    ----------
    df_dms : pd.DataFrame
        Rows of the DMS_MaP experiment (output of load_data_and_encode).
    df_2a3 : pd.DataFrame
        Rows of the 2A3_MaP experiment, in the same order as df_dms.
    maxlen : int
        Number of positions kept for each sequence (206 at most,
        because there are 206 reactivity columns).

    Returns
    -------
    y : np.ndarray of shape (n_samples, maxlen, 2)
        Reactivities clipped between 0 and 1. Index 0 of the last axis is
        DMS_MaP and index 1 is 2A3_MaP (same order as the submission file).
        NaN means "no measure at this position".
    """
    reactivity_cols = []
    for i in range(1, maxlen + 1):
        reactivity_cols.append("reactivity_" + str(i).zfill(4))

    y_dms = df_dms[reactivity_cols].to_numpy()
    y_2a3 = df_2a3[reactivity_cols].to_numpy()

    # Values must be between 0 and 1 (sigmoid output, and same as the Kaggle
    # score). NaN stay NaN.
    y_dms = np.clip(y_dms, 0, 1)
    y_2a3 = np.clip(y_2a3, 0, 1)

    # We put the two measures side by side on the last axis
    y = np.stack([y_dms, y_2a3], axis=-1)

    return y

In [ ]:
def prepare_data(encoded_seqs, y, maxlen):
    """
    Pad the encoded sequences to a fixed length, then split the dataset
    into training and test sets.

    Parameters
    ----------
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    y : np.ndarray of shape (n_samples, maxlen, 2)
        Targets (output of build_targets).
    maxlen : int
        Fixed length to which every sequence is padded or truncated.

    Returns
    -------
    X : np.ndarray of shape (n_samples, maxlen)
        Padded encoded sequences.
    x_train, x_test : np.ndarray
        Train/test split of X.
    y_train, y_test : np.ndarray
        Train/test split of y.
    """
    X = keras.utils.pad_sequences(
        encoded_seqs,
        maxlen=maxlen,
        padding="post",
        truncating="post",
        value=0,
    )

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    return X, x_train, x_test, y_train, y_test

In [ ]:
def create_model(maxlen, embed_dim, vocab_size, num_outputs, num_heads, dense_neuron, num_block_treatment):
    """
    Build the model architecture.

    Parameters
    ----------
    maxlen : int
        Fixed length of the input sequences (after padding/truncation).
    embed_dim : int
        Size of the embeddings (token + position) and attention key size.
    vocab_size : int
        Size of the nucleotide vocabulary (number of distinct indices).
    num_outputs : int
        Number of values predicted for each position (2: DMS_MaP, 2A3_MaP).
    num_heads : int
        Number of attention heads in each MultiHeadAttention block.
    dense_neuron : int
        Number of neurons of the intermediate dense layer (before the output).
    num_block_treatment : int
        Number of residual blocks (normalization + self-attention) stacked.

    Returns
    -------
    model : keras.Model
        Model not compiled, taking a batch of encoded sequences of shape
        (batch_size, maxlen) and giving reactivities between 0 and 1 of
        shape (batch_size, maxlen, num_outputs).
    """
    inputs = keras.Input(shape=(maxlen,))
    x_emb = keras_hub.layers.TokenAndPositionEmbedding(
        vocabulary_size=vocab_size,
        sequence_length=maxlen,
        embedding_dim=embed_dim,
    )(inputs)
    x = x_emb
    for i in range(num_block_treatment):
        x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
        x_atn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x_norm, x_norm)
        x = x + x_atn
    dens = layers.Dense(dense_neuron)(x)
    outputs = layers.Dense(num_outputs, activation="sigmoid")(dens)
    model = keras.Model(inputs=inputs, outputs=outputs)

    return model


In [ ]:
def masked_mae(y_true, y_pred):
    """
    Mean absolute error like the Kaggle score: one single mean over all the
    measured values. The NaN of y_true (padding, no measure) are ignored.

    Parameters
    ----------
    y_true : tensor of shape (batch_size, maxlen, 2)
        True reactivities, NaN where there is no value.
    y_pred : tensor of shape (batch_size, maxlen, 2)
        Reactivities predicted by the model.

    Returns
    -------
    mae : tensor
        One number: the sum of all the errors divided by the number of
        measured values in the batch.
    """
    # mask = 1 if there is a real value, 0 if NaN
    mask = ops.cast(ops.logical_not(ops.isnan(y_true)), "float32")

    # We replace the NaN by the prediction, so the error is 0 at this place
    y_true = ops.where(mask > 0, y_true, y_pred)
    erreur = ops.abs(y_true - y_pred) * mask

    return ops.sum(erreur) / (ops.sum(mask) + 1e-7)

In [ ]:
def train_model(model, x_train, y_train, batch_size, epoch_num, patience):
    """
    Compile and train the model.

    Parameters
    ----------
    model : keras.Model
        Model not compiled to train (output of create_model).
    x_train : np.ndarray
        Encoded and padded sequences, shape (n_samples, maxlen).
    y_train : np.ndarray
        Reactivities with NaN where there is no value,
        shape (n_samples, maxlen, 2).
    batch_size : int
        Size of the batches used during training.
    epoch_num : int
        Maximum number of epochs.
    patience : int
        Number of epochs without improvement of val_loss accepted before
        the early stopping.

    Returns
    -------
    history : keras.callbacks.History
        Training history with the loss (train and validation) of each epoch.
    """
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss=masked_mae,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        x_train,
        y_train,
        batch_size=batch_size,
        epochs=epoch_num,
        validation_split=0.2,
        callbacks=callbacks,
    )
    return history
